In [ ]:
import install_package

install_package.run()

In [ ]:
import os
import time
import math
import torch
import logging
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Dict, List, Optional, Union
from torch.utils.data import Dataset, TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

In [ ]:
# plt.rcParams["font.family"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 12

# 配置日志记录器
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)  # 仅控制台输出
logger = logging.getLogger("Record")

In [ ]:
class GameDataLoader:
    """
    数据加载器类，负责加载和预处理数据集

    功能:
    - 加载所有文件中的数据
    - 返回整合后的数据字典，包括:
        ·'test_id': 测试集，用于预测，仅包含user_id
        ·'level_seq': 包含用户游玩每个关卡的记录，每一条记录是对某个关卡的一次尝试
        ·'level_meta': 每个关卡的一些统计特征，可用于表示关卡
        ·'test_label': 测试集的标签，包含user_id和label
        ·'train_label': 训练集的标签，包含user_id和label
        ·'dev_label': 验证集的标签，包含user_id和label
    """

    def __init__(self, data_dir: str = "./data/") -> None:
        """
        初始化数据加载器

        参数:
            data_dir (str): 数据目录路径，默认为'./data/'
        """

        self.data_dir: str = data_dir
        self.train_label: Optional[pd.DataFrame] = None
        self.dev_label: Optional[pd.DataFrame] = None
        self.test_id: Optional[pd.DataFrame] = None
        self.level_seq: Optional[pd.DataFrame] = None
        self.level_meta: Optional[pd.DataFrame] = None
        self.test_label: Optional[pd.DataFrame] = None

    def _get_data(self) -> None:
        """
        加载所有数据文件
        """

        logger.info("开始加载所有数据集...")
        try:
            # 加载训练集标签、验证集标签和测试集user_id
            self.train_label = pd.read_csv(self.data_dir + "train.csv", sep="\t")
            self.dev_label = pd.read_csv(self.data_dir + "dev.csv", sep="\t")
            self.test_id = pd.read_csv(self.data_dir + "test.csv", sep="\t")
            # 加载游戏记录和关卡元数据
            self.level_seq = pd.read_csv(self.data_dir + "level_seq.csv", sep="\t")
            self.level_meta = pd.read_csv(self.data_dir + "level_meta.csv", sep="\t")
            # 加载测试集真实标签
            self.test_label = pd.read_csv(self.data_dir + "Groundtruth.csv", sep=",")
            self.test_label = self.test_label.rename(
                columns={"ID": "user_id", "Label": "label"}
            )
            # 转换时间列
            self.level_seq["time"] = pd.to_datetime(
                self.level_seq["time"], errors="coerce"
            )

            logger.info("所有数据加载完成")

        except Exception as e:
            logger.error(f"加载数据时出错: {str(e)}")
            raise

    def _integrate_data(self) -> Dict[str, pd.DataFrame]:
        """
        整合所有数据集并返回一个包含所有数据集的字典

        返回:
            Dict[str, pd.DataFrame]: 包含以下键的字典:
                - 'train_label': 训练集user_id和标签
                - 'dev_label': 验证集user_id和标签
                - 'test_id': 测试集user_id
                - 'level_seq': 游戏行为序列
                - 'level_meta': 关卡元数据
                - 'test_label': 测试集user_id标签
        """

        logger.info("开始整合数据集...")

        datasets = {
            "train_label": self.train_label,
            "dev_label": self.dev_label,
            "test_id": self.test_id,
            "level_seq": self.level_seq,
            "level_meta": self.level_meta,
            "test_label": self.test_label,
        }

        logger.info("数据集整合完成")

        return datasets

    def load_all_data(self) -> Dict[str, pd.DataFrame]:
        """
        加载并返回所有数据

        返回:
            Dict[str, pd.DataFrame]: 整合后的数据集字典
        """

        self._get_data()
        return self._integrate_data()

In [ ]:
class SequenceDataProcessor:
    """
    序列数据处理类，负责将原始数据转换为序列格式
    """

    def __init__(
        self,
        level_seq: pd.DataFrame,
        level_meta: pd.DataFrame,
        max_seq_len: int = 100,
        use_normalization: bool = True,
    ) -> None:
        """
        初始化序列处理器

        参数:
            level_seq (pd.DataFrame): 包含用户游玩记录的DataFrame
            level_meta (pd.DataFrame): 关卡元数据
            max_seq_len (int): 最大序列长度，默认为100
        """

        self.level_seq = level_seq
        self.level_meta = level_meta
        self.max_seq_len = max_seq_len
        self.feature_names = None
        self.use_normalization = use_normalization
        self.scaler = None

        # 合并关卡元数据
        self.merged_seq = pd.merge(level_seq, level_meta, on="level_id", how="left")

        # 按时间和用户排序
        self.merged_seq["time"] = pd.to_datetime(self.merged_seq["time"])
        self.merged_seq = self.merged_seq.sort_values(["user_id", "time"])

    def extract_features(self) -> pd.DataFrame:
        """
        提取用于序列建模的特征
        """

        # 基本特征
        features = [
            "f_success",
            "f_duration",
            "f_reststep",
            "f_help",
            "f_avg_duration",
            "f_avg_passrate",
            "f_avg_retrytimes",
        ]

        # 衍生特征
        self.merged_seq["time_diff"] = (
            self.merged_seq.groupby("user_id")["time"].diff().dt.total_seconds()
        )
        self.merged_seq["prev_success"] = self.merged_seq.groupby("user_id")[
            "f_success"
        ].shift(1)
        self.merged_seq["duration_ratio"] = (
            self.merged_seq["f_duration"] / self.merged_seq["f_avg_duration"]
        )

        # 填充第一个时间点的缺失值
        self.merged_seq["time_diff"] = self.merged_seq["time_diff"].fillna(0)
        self.merged_seq["prev_success"] = self.merged_seq["prev_success"].fillna(0)

        # 添加特征名
        self.feature_names = features + ["time_diff", "prev_success", "duration_ratio"]

        if self.use_normalization:
            if self.scaler is None:
                self.scaler = StandardScaler()
                self.scaler.fit(self.merged_seq[self.feature_names])
            normalized_features = self.scaler.transform(
                self.merged_seq[self.feature_names]
            )
            for i, col in enumerate(self.feature_names):
                self.merged_seq[col] = normalized_features[:, i]

        return self.merged_seq[self.feature_names + ["user_id"]]

    def create_sequences(self, user_labels: pd.DataFrame) -> tuple:
        """
        创建序列数据集
        """

        # 提取特征
        feature_df = self.extract_features()

        # 获取用户列表
        users = feature_df["user_id"].unique()

        # 初始化存储
        sequences = []
        labels = []
        sequence_lengths = []

        # 为每个用户创建序列
        for user in users:
            user_data = feature_df[feature_df["user_id"] == user]
            user_features = user_data[self.feature_names].values

            # 截断或填充序列
            if len(user_features) > self.max_seq_len:
                seq = user_features[-self.max_seq_len :]
                seq_len = self.max_seq_len
            else:
                padding = np.zeros(
                    (self.max_seq_len - len(user_features), len(self.feature_names))
                )
                seq = np.vstack([padding, user_features])
                seq_len = len(user_features)

            sequences.append(seq)
            sequence_lengths.append(seq_len)

            # 获取用户标签
            user_label = user_labels[user_labels["user_id"] == user]["label"].values
            if len(user_label) > 0:
                labels.append(user_label[0])
            else:
                labels.append(0)

        sequences = np.array(sequences)
        labels = np.array(labels)
        sequence_lengths = np.array(sequence_lengths)

        # logger.info(f"创建了{len(sequences)}个序列，序列形状: {sequences.shape}")

        return sequences, labels, sequence_lengths, self.feature_names

In [ ]:
class SequenceModel(nn.Module):
    """
    序列模型基类
    """

    def __init__(
        self, input_size: int, hidden_size: int, num_layers: int, dropout: float = 0.3
    ):
        super(SequenceModel, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout

        # LSTM层
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        # 输出层
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x, sequence_lengths=None):
        # 初始化隐藏状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # 打包序列（处理变长序列）
        if sequence_lengths is not None:
            x = nn.utils.rnn.pack_padded_sequence(
                x, sequence_lengths.cpu(), batch_first=True, enforce_sorted=False
            )

        # LSTM前向传播
        out, _ = self.lstm(x, (h0, c0))

        # 解包序列
        if sequence_lengths is not None:
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

        # 获取序列最后一个时间步的输出
        if sequence_lengths is not None:
            # 对于每个序列，获取其实际最后一个时间步的输出
            idx = (
                (sequence_lengths - 1)
                .view(-1, 1)
                .expand(-1, self.hidden_size)
                .unsqueeze(1)
            )
            last_out = out.gather(1, idx).squeeze(1)
        else:
            last_out = out[:, -1, :]

        # 全连接层
        output = self.fc(last_out)
        return output

In [ ]:
class GRUModel(nn.Module):
    """
    GRU序列模型
    """

    def __init__(
        self, input_size: int, hidden_size: int, num_layers: int, dropout: float = 0.3
    ):
        super(GRUModel, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout

        # GRU层
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        # 输出层
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x, sequence_lengths=None):
        # 初始化隐藏状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # 打包序列（处理变长序列）
        if sequence_lengths is not None:
            x = nn.utils.rnn.pack_padded_sequence(
                x, sequence_lengths.cpu(), batch_first=True, enforce_sorted=False
            )

        # GRU前向传播
        out, _ = self.gru(x, h0)

        # 解包序列
        if sequence_lengths is not None:
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

        # 获取序列最后一个时间步的输出
        if sequence_lengths is not None:
            # 对于每个序列，获取其实际最后一个时间步的输出
            idx = (
                (sequence_lengths - 1)
                .view(-1, 1)
                .expand(-1, self.hidden_size)
                .unsqueeze(1)
            )
            last_out = out.gather(1, idx).squeeze(1)
        else:
            last_out = out[:, -1, :]

        # 全连接层
        output = self.fc(last_out)
        return output


In [ ]:
class PositionalEncoding(nn.Module):
    """
    位置编码层
    """

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 计算位置编码
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)

In [ ]:
class Lambda(nn.Module):
    def __init__(self, func):
        super().__init__()
        self.func = func

    def forward(self, x):
        return self.func(x)

In [ ]:
class TransformerModel(nn.Module):
    """
    Transformer序列模型
    """

    def __init__(
        self,
        input_size: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dropout: float = 0.3,
    ):
        super(TransformerModel, self).__init__()
        self.d_model = d_model

        # 输入投影层
        self.input_proj = nn.Linear(input_size, d_model)

        # 位置编码
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # Transformer编码器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model,
            nhead,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers, enable_nested_tensor=False
        )

        # 输出层
        self.fc = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid(),
            Lambda(lambda x: torch.clamp(x, 1e-7, 1 - 1e-7)),
        )

    def forward(
        self, x: torch.Tensor, sequence_lengths: torch.Tensor = None
    ) -> torch.Tensor:
        # 输入投影 [batch, seq_len, input_size] -> [batch, seq_len, d_model]
        x = self.input_proj(x)

        # 缩放嵌入
        x = x * math.sqrt(self.d_model)

        # 添加位置编码
        x = self.pos_encoder(x)

        # 创建padding mask
        padding_mask = None
        if sequence_lengths is not None:
            max_len = x.size(1)
            padding_mask = torch.arange(max_len, device=x.device).expand(
                len(sequence_lengths), max_len
            ) >= sequence_lengths.unsqueeze(1)

        # Transformer编码器
        output = self.transformer_encoder(x, src_key_padding_mask=padding_mask)

        # 获取序列最后一个有效时间步的输出
        if sequence_lengths is not None:
            # 更高效地提取最后一个有效时间步
            last_indices = sequence_lengths - 1
            last_out = output[torch.arange(output.size(0)), last_indices]
        else:
            last_out = output[:, -1, :]

        # 全连接层
        return self.fc(last_out)

In [ ]:
class SequenceTrainer:
    """
    序列模型训练器
    """

    def __init__(self, model, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.BCELoss()
        self.optimizer = optim.Adam(model.parameters(), lr=0.001)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="max", factor=0.1, patience=5
        )

    def train(self, train_loader, val_loader=None, epochs=20, early_stop_patience=5):
        """
        训练模型
        """

        best_val_auc = 0
        no_improve_epochs = 0
        train_history = {"loss": [], "auc": []}
        val_history = {"loss": [], "auc": []}

        for epoch in range(epochs):
            self.model.train()
            epoch_loss = 0
            all_labels = []
            all_preds = []

            # 训练阶段
            for sequences, lengths, labels in tqdm(
                train_loader, desc=f"Epoch {epoch + 1}/{epochs}"
            ):
                sequences, labels = sequences.to(self.device), labels.to(self.device)
                if lengths is not None:
                    lengths = lengths.to(self.device)

                # 前向传播
                outputs = self.model(sequences, lengths)
                loss = self.criterion(outputs.squeeze(), labels.float())

                # 反向传播
                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                # 记录损失和预测
                epoch_loss += loss.item()
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(outputs.detach().cpu().numpy())

            # 计算训练指标
            train_loss = epoch_loss / len(train_loader)
            train_auc = roc_auc_score(all_labels, all_preds)
            train_history["loss"].append(train_loss)
            train_history["auc"].append(train_auc)

            # 验证阶段
            if val_loader is not None:
                val_loss, val_auc = self.evaluate(val_loader)
                val_history["loss"].append(val_loss)
                val_history["auc"].append(val_auc)

                # 更新学习率
                self.scheduler.step(val_auc)

                logger.info(
                    f"Epoch {epoch + 1}/{epochs}: "
                    f"Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f}, "
                    f"Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}"
                )

                # 早停检查
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    no_improve_epochs = 0
                    torch.save(self.model.state_dict(), "best_model.pth")
                else:
                    no_improve_epochs += 1
                    if no_improve_epochs >= early_stop_patience:
                        logger.info(f"Early stopping at epoch {epoch + 1}")
                        break
            else:
                logger.info(
                    f"Epoch {epoch + 1}/{epochs}: "
                    f"Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f}"
                )

        # 加载最佳模型
        if val_loader is not None and os.path.exists("best_model.pth"):
            self.model.load_state_dict(torch.load("best_model.pth"))

        return train_history, val_history

    def evaluate(self, test_loader):
        """
        评估模型
        """

        self.model.eval()
        test_loss = 0
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for sequences, lengths, labels in test_loader:
                sequences, labels = sequences.to(self.device), labels.to(self.device)
                if lengths is not None:
                    lengths = lengths.to(self.device)

                outputs = self.model(sequences, lengths)
                loss = self.criterion(outputs.squeeze(), labels.float())

                test_loss += loss.item()
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(outputs.cpu().numpy())

        test_loss /= len(test_loader)
        test_auc = roc_auc_score(all_labels, all_preds)

        return test_loss, test_auc

    def predict(self, test_loader):
        """
        生成预测
        """

        self.model.eval()
        all_preds = []

        with torch.no_grad():
            for sequences, lengths, _ in test_loader:
                sequences = sequences.to(self.device)
                if lengths is not None:
                    lengths = lengths.to(self.device)

                outputs = self.model(sequences, lengths)
                all_preds.extend(outputs.cpu().numpy())

        return np.array(all_preds).squeeze()

In [ ]:
def create_data_loaders(sequences, labels, lengths, batch_size=32, shuffle=True):
    """
    创建PyTorch数据加载器
    """

    # 转换为张量
    sequences_tensor = torch.tensor(sequences, dtype=torch.float32)
    labels_tensor = torch.tensor(labels, dtype=torch.float32)
    lengths_tensor = torch.tensor(lengths, dtype=torch.long)

    # 创建数据集
    dataset = TensorDataset(sequences_tensor, lengths_tensor, labels_tensor)

    # 创建数据加载器
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=lambda batch: (
            torch.stack([item[0] for item in batch]),
            torch.stack([item[1] for item in batch]),
            torch.stack([item[2] for item in batch]),
        ),
    )

    return loader

In [ ]:
def plot_training_history(name, train_history, val_history=None):
    """
    绘制训练历史
    """

    epochs = len(train_history["loss"])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{name} training history curve", color="red")

    # 损失曲线 - 左图
    ax1.plot(list(range(1, epochs + 1)), train_history["loss"], label="Training Loss")
    ax1.set_title("Loss over Epochs", color="purple", fontsize=16)
    ax1.set_xlabel("Epochs", color="blue")
    ax1.set_ylabel("Loss", color="blue")
    ax1.set_xticks(range(1, epochs + 1))
    ax1.grid(True, linestyle="--", alpha=0.2)
    ax1.legend()
    ax1.margins(x=0.05, y=0.12)

    # AUC曲线 - 右图
    ax2.plot(list(range(1, epochs + 1)), train_history["auc"], label="Training AUC")
    ax2.set_title("AUC over Epochs", color="purple")
    ax2.set_xlabel("Epochs", color="blue")
    ax2.set_ylabel("AUC", color="blue")
    ax2.set_xticks(range(1, epochs + 1))
    ax2.grid(True, linestyle="--", alpha=0.2)
    ax2.legend()
    ax2.margins(x=0.05, y=0.12)

    plt.tight_layout()
    plt.show()

In [ ]:
def run_sequence_experiment(data_dir: str = "./data/", max_seq_len: int = 100):
    """
    运行序列模型实验
    """

    logger.info("开始序列模型实验...")

    # 1. 加载数据
    logger.info("加载数据...")
    data_loader = GameDataLoader(data_dir)
    datasets = data_loader.load_all_data()

    # 2. 准备序列数据
    logger.info("准备序列数据...")
    processor = SequenceDataProcessor(
        datasets["level_seq"], datasets["level_meta"], max_seq_len=max_seq_len
    )

    train_sequences, train_labels, train_lengths, feature_names = (
        processor.create_sequences(
            pd.concat([datasets["train_label"], datasets["dev_label"]])
        )
    )

    test_sequences, test_labels, test_lengths, _ = processor.create_sequences(
        datasets["test_label"]
    )

    logger.info(f"特征数量: {len(feature_names)}")
    logger.info(
        f"训练序列形状: {train_sequences.shape}, 测试序列形状: {test_sequences.shape}"
    )

    # 3. 创建数据加载器
    logger.info("创建数据加载器...")
    train_loader = create_data_loaders(
        train_sequences, train_labels, train_lengths, batch_size=32
    )
    test_loader = create_data_loaders(
        test_sequences, test_labels, test_lengths, batch_size=32, shuffle=False
    )

    # 4. 训练LSTM模型
    logger.info("训练LSTM模型...")
    lstm_model = SequenceModel(
        input_size=len(feature_names), hidden_size=64, num_layers=2, dropout=0.3
    )

    lstm_trainer = SequenceTrainer(lstm_model)
    lstm_train_history, _ = lstm_trainer.train(train_loader)

    # 评估LSTM模型
    lstm_test_loss, lstm_test_auc = lstm_trainer.evaluate(test_loader)
    logger.info(
        f"LSTM模型测试结果 - Loss: {lstm_test_loss:.4f}, AUC: {lstm_test_auc:.4f}"
    )

    # 5. 训练GRU模型
    logger.info("训练GRU模型...")
    gru_model = GRUModel(
        input_size=len(feature_names), hidden_size=64, num_layers=2, dropout=0.3
    )

    gru_trainer = SequenceTrainer(gru_model)
    gru_train_history, _ = gru_trainer.train(train_loader)

    # 评估GRU模型
    gru_test_loss, gru_test_auc = gru_trainer.evaluate(test_loader)
    logger.info(f"GRU模型测试结果 - Loss: {gru_test_loss:.4f}, AUC: {gru_test_auc:.4f}")

    # 6. 训练Transformer模型
    logger.info("训练Transformer模型...")
    transformer_model = TransformerModel(
        input_size=len(feature_names), d_model=64, nhead=4, num_layers=2, dropout=0.3
    )

    transformer_trainer = SequenceTrainer(transformer_model)
    transformer_train_history, _ = transformer_trainer.train(train_loader)

    # 评估Transformer模型
    transformer_test_loss, transformer_test_auc = transformer_trainer.evaluate(
        test_loader
    )
    logger.info(
        f"Transformer模型测试结果 - Loss: {transformer_test_loss:.4f}, AUC: {transformer_test_auc:.4f}"
    )

    # 7. 可视化训练过程
    plot_training_history("LSTM", lstm_train_history)
    plot_training_history("GRU", gru_train_history)
    plot_training_history("Transformer", transformer_train_history)

    # 8. 生成最终预测
    logger.info("生成预测结果...")
    lstm_preds = lstm_trainer.predict(test_loader)
    gru_preds = gru_trainer.predict(test_loader)
    transformer_preds = transformer_trainer.predict(test_loader)

    # 融合预测（加权平均）
    ensemble_preds = lstm_preds * 0.3 + transformer_preds * 0.4 + gru_preds * 0.3

    # 计算融合模型AUC
    ensemble_auc = roc_auc_score(test_labels, ensemble_preds)
    logger.info(f"融合模型AUC: {ensemble_auc:.4f}")

    # 返回结果
    results = {
        "LSTM": {"AUC": lstm_test_auc},
        "GRU": {"AUC": gru_test_auc},
        "Transformer": {"AUC": transformer_test_auc},
        "Ensemble": {"AUC": ensemble_auc},
        "feature_names": feature_names,
    }

    logger.info("序列模型实验完成!")
    return results

In [ ]:
sequence_results = run_sequence_experiment()

### 序列建模实验总结

#### **1. 数据集特点**

- **数据规模**

  - 训练集：13,589 个用户序列
  - 测试集：13,589 个用户序列
  - 序列长度：固定为 1,000（通过截断/填充实现）
  - 特征维度：10 维（7 基础特征+3 衍生特征）

- **预处理流程**

1. 合并关卡元数据与行为序列
2. 按用户和时间排序
3. 特征标准化（`StandardScaler`）
4. 序列对齐（截断长序列，零填充短序列）

---

#### **2. 模型架构对比**

| **模型类型**    | **核心结构** | **隐藏层** | **处理机制**          | **计算效率**    |
| --------------- | ------------ | ---------- | --------------------- | --------------- |
| **LSTM**        | 2 层 LSTM    | 64 维      | 变长序列打包          | 高（27 it/s）   |
| **GRU**         | 2 层 GRU     | 64 维      | 变长序列打包          | 最高（28 it/s） |
| **Transformer** | 2 层编码器   | 64 维      | 位置编码+padding mask | 低（9 it/s）    |

---

#### **3. 训练过程分析**

- **超参数配置**：
  - 批次大小：32
  - 训练轮次：20 epochs
  - Dropout 率：0.3

---

- **关键观察**：

1. GRU 收敛速度最快（第 5 轮即达 0.71+ AUC）
2. Transformer 前期提升明显（Epoch1→Epoch3: +0.0316 AUC）
3. 所有模型在 10 轮后进入平台期

---

#### **4. 实验结果对比**

| **评估指标**       | LSTM   | GRU    | Transformer | 融合模型 |
| ------------------ | ------ | ------ | ----------- | -------- |
| **测试 Loss**      | 0.4097 | 0.3833 | 0.3850      | -        |
| **测试 AUC**       | 0.6691 | 0.6678 | 0.6686      | 0.6804   |
| **训练 AUC**       | 0.7167 | 0.7148 | 0.7139      | -        |
| **训练时间/epoch** | 15s    | 15s    | 45s         | -        |

---

**核心结论**：

1. 存在明显过拟合（训练 AUC 比测试 AUC 高 4.5-5%）
2. Transformer 测试表现最优（AUC 0.6686）
3. 加权融合显著提升性能（+0.0118 AUC）

---

#### **5. 问题诊断与改进建议**

**主要问题**：

- 过拟合风险（训练/测试 AUC 差距大）
- 长序列处理效率低（1000 步固定长度）
- Transformer 计算成本过高

**可能的改进**：

1. 将最大序列长度从 1000 缩减至 200（保留近期关键行为）
2. 添加 Layer Normalization 提升 Transformer 稳定性
3. 使用早停机制（当验证 AUC 连续 3 轮不提升时终止训练）
